# Aswenna Agricultural Marketplace - Tabular Modeling Pipeline

This notebook covers the complete end-to-end machine learning pipeline for the **Aswenna Agricultural Marketplace** tabular datasets:
1. **Data Preprocessing & Merging**
2. **Exploratory Data Analysis (EDA) & Charts**
3. **Model Training & Evaluation**:
   - **Task A: Crop Recommendation** (Multi-class classification predicting the optimal crop based on soil and climate conditions)
   - **Task B: Crop Yield Forecasting** (Regression predicting yield in hg/ha based on climate, crop type, and pesticide inputs)
   - **Task C: Fertilizer Recommendation** (Multi-class classification predicting the best fertilizer for soil configurations)
4. **Model Serialization** (Saving trained models for API production usage)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score, confusion_matrix

# Create directories for processed datasets, plots, and models
os.makedirs('datasets/processed', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("Libraries loaded and directories initialized.")

Libraries loaded and directories initialized.


## 1. Load Raw Datasets

In [2]:
# Load datasets
crop_rec_df = pd.read_csv('datasets/raw/Crop Recommendation Dataset/Crop_Recommendation.csv')
crop_yield_df = pd.read_csv('datasets/raw/Crop Yield Prediction Dataset/yield_df.csv')
crop_soil_df = pd.read_csv('datasets/raw/Crop and Soil DataSet/data_core.csv')

print(f"Crop Recommendation shape: {crop_rec_df.shape}")
print(f"Crop Yield shape: {crop_yield_df.shape}")
print(f"Crop Soil shape: {crop_soil_df.shape}")

Crop Recommendation shape: (2200, 8)
Crop Yield shape: (28242, 8)
Crop Soil shape: (8000, 9)


## 2. Preprocessing & Data Merging

We will preprocess each dataset individually and merge them where relevant (saving the cleaned versions in `datasets/processed`).

In [3]:
# -----------------------------------------------------
# 2.1 Crop Recommendation Dataset Cleaning
# -----------------------------------------------------
crop_rec_df = crop_rec_df.dropna()
crop_rec_df.columns = [col.strip() for col in crop_rec_df.columns]
crop_rec_df.to_csv('datasets/processed/cleaned_crop_recommendation.csv', index=False)

# -----------------------------------------------------
# 2.2 Crop Yield Dataset Cleaning
# -----------------------------------------------------
if 'Unnamed: 0' in crop_yield_df.columns:
    crop_yield_df = crop_yield_df.drop(columns=['Unnamed: 0'])
crop_yield_df = crop_yield_df.dropna()
crop_yield_df.to_csv('datasets/processed/cleaned_crop_yield.csv', index=False)

# -----------------------------------------------------
# 2.3 Crop Soil & Fertilizer Cleaning
# -----------------------------------------------------
crop_soil_df = crop_soil_df.dropna()
crop_soil_df.columns = [col.strip() for col in crop_soil_df.columns]
crop_soil_df.to_csv('datasets/processed/cleaned_crop_soil.csv', index=False)

print("Datasets preprocessed and saved to datasets/processed/.")

Datasets preprocessed and saved to datasets/processed/.


## 3. Exploratory Data Analysis & Visualizations

We will generate exploratory charts (correlation matrix and distribution profiles) and save them to `plots/`.

In [4]:
# 3.1 Heatmap for Crop Recommendation features
plt.figure(figsize=(10, 8))
numeric_cols = crop_rec_df.select_dtypes(include=[np.number]).columns
sns.heatmap(crop_rec_df[numeric_cols].corr(), annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation Matrix - Crop Recommendation Features')
plt.tight_layout()
plt.savefig('plots/crop_recommendation_correlation.png', dpi=300)
plt.close()

# 3.2 Heatmap for Fertilizer Recommendation features
plt.figure(figsize=(10, 8))
numeric_soil_cols = crop_soil_df.select_dtypes(include=[np.number]).columns
sns.heatmap(crop_soil_df[numeric_soil_cols].corr(), annot=True, cmap='Oranges', fmt='.2f')
plt.title('Correlation Matrix - Crop Soil & Fertilizer Features')
plt.tight_layout()
plt.savefig('plots/crop_soil_correlation.png', dpi=300)
plt.close()

# 3.3 Yield Distribution Plot
plt.figure(figsize=(10, 6))
sns.histplot(crop_yield_df['hg/ha_yield'], bins=50, kde=True, color='green')
plt.title('Crop Yield Distribution (hg/ha)')
plt.xlabel('Yield (hg/ha)')
plt.tight_layout()
plt.savefig('plots/crop_yield_distribution.png', dpi=300)
plt.close()

print("Visualizations generated and saved to plots/.")

Visualizations generated and saved to plots/.


## 4. Task A: Crop Recommendation (Classification)

Determining the optimal crop based on Nitrogen, Phosphorus, Potassium, Temperature, Humidity, pH, and Rainfall.

In [5]:
X_rec = crop_rec_df.drop(columns=['Crop'])
y_rec = crop_rec_df['Crop']

# Encode targets
le_rec = LabelEncoder()
y_rec_encoded = le_rec.fit_transform(y_rec)

X_train, X_test, y_train, y_test = train_test_split(X_rec, y_rec_encoded, test_size=0.2, random_state=42, stratify=y_rec_encoded)

print("Training LightGBM Classifier...")
clf_lgb = LGBMClassifier(random_state=42, verbose=-1)
clf_lgb.fit(X_train, y_train)

y_pred = clf_lgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"LightGBM Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le_rec.classes_))

# Save Crop Recommendation Model & Label Encoder
joblib.dump(clf_lgb, 'models/crop_recommendation_model.pkl')
joblib.dump(le_rec, 'models/crop_recommendation_encoder.pkl')

Training LightGBM Classifier...
LightGBM Accuracy: 0.9886

Classification Report:
              precision    recall  f1-score   support

       Apple       1.00      1.00      1.00        20
      Banana       1.00      1.00      1.00        20
   Blackgram       0.95      1.00      0.98        20
    ChickPea       1.00      1.00      1.00        20
     Coconut       1.00      1.00      1.00        20
      Coffee       1.00      1.00      1.00        20
      Cotton       1.00      1.00      1.00        20
      Grapes       1.00      1.00      1.00        20
        Jute       0.91      1.00      0.95        20
 KidneyBeans       0.95      1.00      0.98        20
      Lentil       1.00      0.95      0.97        20
       Maize       1.00      1.00      1.00        20
       Mango       1.00      1.00      1.00        20
   MothBeans       0.95      1.00      0.98        20
    MungBean       1.00      1.00      1.00        20
   Muskmelon       1.00      1.00      1.00        20

['models/crop_recommendation_encoder.pkl']

## 5. Task B: Crop Yield Forecasting (Regression)

Predicting crop yield based on Area, Item (crop), Year, Average Rainfall, Pesticide tonnage, and Average Temperature.

In [6]:
X_yield = crop_yield_df.drop(columns=['hg/ha_yield'])
y_yield = crop_yield_df['hg/ha_yield']

# Encode categorical variables (Area & Item)
le_area = LabelEncoder()
le_item = LabelEncoder()

X_yield['Area'] = le_area.fit_transform(X_yield['Area'])
X_yield['Item'] = le_item.fit_transform(X_yield['Item'])

X_train_y, X_test_y, y_train_y, y_test_y = train_test_split(X_yield, y_yield, test_size=0.2, random_state=42)

print("Training XGBoost Regressor...")
reg_xgb = XGBRegressor(n_estimators=150, learning_rate=0.08, random_state=42)
reg_xgb.fit(X_train_y, y_train_y)

y_pred_y = reg_xgb.predict(X_test_y)
rmse = np.sqrt(mean_squared_error(y_test_y, y_pred_y))
r2 = r2_score(y_test_y, y_pred_y)

print(f"XGBoost RMSE: {rmse:.4f}")
print(f"XGBoost R^2 Score: {r2:.4f}")

# Save Crop Yield Model & Encoders
joblib.dump(reg_xgb, 'models/crop_yield_model.pkl')
joblib.dump(le_area, 'models/crop_yield_area_encoder.pkl')
joblib.dump(le_item, 'models/crop_yield_item_encoder.pkl')

Training XGBoost Regressor...
XGBoost RMSE: 16089.7378
XGBoost R^2 Score: 0.9643


['models/crop_yield_item_encoder.pkl']

## 6. Task C: Fertilizer Recommendation (Classification)

Predicting the best fertilizer name based on soil characteristics, NPK composition, soil type, and crop type.

In [7]:
X_soil = crop_soil_df.drop(columns=['Fertilizer Name'])
y_soil = crop_soil_df['Fertilizer Name']

# Encode categoricals
le_soil_type = LabelEncoder()
le_crop_type = LabelEncoder()
le_fertilizer = LabelEncoder()

X_soil['Soil Type'] = le_soil_type.fit_transform(X_soil['Soil Type'])
X_soil['Crop Type'] = le_crop_type.fit_transform(X_soil['Crop Type'])
y_soil_encoded = le_fertilizer.fit_transform(y_soil)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_soil, y_soil_encoded, test_size=0.2, random_state=42, stratify=y_soil_encoded)

print("Training Random Forest Classifier...")
clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf.fit(X_train_s, y_train_s)

y_pred_s = clf_rf.predict(X_test_s)
acc_s = accuracy_score(y_test_s, y_pred_s)
print(f"Random Forest Fertilizer Accuracy: {acc_s:.4f}")

# Save Fertilizer recommendation models & encoders
joblib.dump(clf_rf, 'models/fertilizer_recommendation_model.pkl')
joblib.dump(le_soil_type, 'models/fertilizer_soil_encoder.pkl')
joblib.dump(le_crop_type, 'models/fertilizer_crop_encoder.pkl')
joblib.dump(le_fertilizer, 'models/fertilizer_name_encoder.pkl')

print("\nAll models and encoders serialized successfully.")

Training Random Forest Classifier...
Random Forest Fertilizer Accuracy: 0.1519

All models and encoders serialized successfully.
